# Triage dermatológico con IA — demo en vivo
### SI4006 · Proyecto Integrador · Universidad EAFIT

Herramienta de **apoyo a la decisión** para personal médico de triage: a partir del síntoma
descrito (y una foto opcional de la lesión) sugiere si el caso conviene **referirlo con
prioridad a dermatología** o puede esperar una **cita de rutina**. No diagnostica; la
decisión es del profesional.

Este notebook levanta la app completa en una GPU T4 de Kaggle y te da un link público
(`https://....gradio.live`) para usarla desde el computador o el celular. Combina:
- un clasificador de texto (BETO) entrenado para urgencia,
- búsqueda en guías clínicas (RAG),
- un clasificador visual para la foto,
- un agente que decide qué herramientas usar, con una regla de seguridad: si alguna señal
  indica urgencia, se sugiere referir.

**Cómo correrlo (unos 3-5 minutos):**
1. Haz clic en **Copy & Edit** (arriba a la derecha) para tener tu propia copia.
2. En el panel derecho, *Session options*: **Accelerator → GPU T4** e **Internet → On**
   (Kaggle pide una cuenta verificada por teléfono para activar internet).
3. **Run All**. La última celda imprime el link `gradio.live`; ábrelo y prueba.

Cuando termines, detén la sesión (*Stop session*) para no gastar tu cuota semanal de GPU.

Código: [github.com/Isa-Idarraga/triage-dermatologico-ia](https://github.com/Isa-Idarraga/triage-dermatologico-ia)

## 1 · Preparar el entorno

In [ ]:
import os

RAMA = "main"

if os.path.isdir("triage-dermatologico-ia"):
    %cd triage-dermatologico-ia
    !git checkout {RAMA}
    !git pull origin {RAMA}
else:
    !git clone --branch {RAMA} https://github.com/Isa-Idarraga/triage-dermatologico-ia.git
    %cd triage-dermatologico-ia

!git log -1 --oneline  # confirmar a simple vista qué commit quedó activo


In [ ]:
%pip install -q -r requirements.txt

# Mismo arreglo que en los notebooks de M2/M3/M4: el entorno trae una version vieja
# de torchao preinstalada que hace tronar a `peft` al importarse.
!pip uninstall -y -q torchao


In [ ]:
import torch
print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("ADVERTENCIA: no hay GPU activa. Ve a Settings -> Accelerator -> GPU T4, "
          "y vuelve a correr esta celda.")


## 2 · Cargar la app y calentar los modelos

`_calentar()` carga los 4 modelos (agente, BETO, RAG y el clasificador visual) una sola
vez, para que la primera consulta no tenga que esperar los ~160 s de arranque en frío.

Dos ajustes opcionales antes de importar la app:
- `TRIAGE_CUANT=fp16`: precisión con la que corren los modelos. Con fp16 la calidad es la
  misma y la consulta tarda unos 12 s en vez de 17 s (ver `m5/eficiencia.md`).
- `WANDB_API_KEY`: solo si quieres registrar la corrida en Weights & Biases. Agrégala en
  *Add-ons → Secrets* de Kaggle, nunca en el código. Si no está, la app funciona igual.

`health()` reporta el estado: `"calentando"` antes de cargar los modelos y `"ok"` después.

In [ ]:
import os

os.environ["TRIAGE_CUANT"] = "fp16"  # ver m5/eficiencia.md

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    print("WANDB_API_KEY cargada desde los Secrets de Kaggle.")
except Exception as e:
    print(f"Sin W&B esta corrida (sigue funcionando igual): {e!r}")

import sys
sys.path.insert(0, "m5/app")
import app

print("Versión (tag de Git):", app._version_tag())
print("health() antes de calentar:", app.health())

app._calentar()

print("health() después de calentar:", app.health())


## 3 · Comprobar que el sistema responde como en la evaluación

Antes de abrir la interfaz, se corren casos ya conocidos de la evaluación del proyecto y se
comprueba con `assert` que el resultado es el mismo. Si algún `assert` falla, el entorno no
quedó bien (por ejemplo, sin GPU o con otra versión de las librerías).
- **Sin foto**: un síntoma de alarma que el texto y las guías marcan como urgente.
- **Con foto**: `ISIC_0024876.jpg` (un lunar benigno, confirmado por biopsia) con un síntoma
  vago. La foto y las guías dicen "no urgente", pero el clasificador de texto dice
  "urgente", y la regla de seguridad sugiere referir: el sistema prefiere una consulta de
  más a dejar pasar un caso grave.

In [ ]:
decision, bloque_foto, evidencia = app.consultar(
    "Tengo un lunar que me cambió de color y de forma en los últimos meses.", None,
)
print(decision)
print()
print(bloque_foto)

assert "Referir con prioridad" in decision, f"se esperaba urgente (BETO y RAG deberian coincidir), salio: {decision}"
assert "No evaluable: no se usó" in bloque_foto, "sin foto, el bloque de foto debe decir que no se uso"
print()
print("OK: coincide con la corrida de referencia en Kaggle T4.")


In [ ]:
_imagen_prueba = "m4/pipeline/eval_tool/imagenes/ISIC_0024876.jpg"
assert os.path.exists(_imagen_prueba), "no se encontro la imagen de prueba del eval_tool"

decision, bloque_foto, evidencia = app.consultar(
    "Le adjunto una foto de la lesión, ¿debería verla pronto?", _imagen_prueba,
)
print(decision)
print()
print(bloque_foto)
print()
print(evidencia)

# Valores de referencia: misma imagen, mismo texto exacto (los acentos importan --
# BETO y el RAG son sensibles al texto) ya confirmados en Colab (M4) y en Kaggle.
assert "Referir con prioridad" in decision, f"la regla de seguridad deberia escalar a urgente, salio: {decision}"
assert "ver_imagen=no_urgente" in decision, f"ver_imagen deberia decir no_urgente en esta imagen: {decision}"
assert "clasificador_beto=urgente" in decision, f"BETO deberia forzar urgente con un sintoma vago: {decision}"
assert "0.157" in bloque_foto, f"el score de ver() deberia ser ~0.1565 (0.157 redondeado): {bloque_foto}"
assert "no describe su foto" in evidencia.lower(), "falta la nota de que la evidencia del RAG no describe la foto"
print()
print("OK: coincide con la corrida de referencia en Kaggle T4 y con el hallazgo de M4.")


### Caso borde: una foto que no se puede evaluar

`img_06.jpg` está desenfocada a propósito. Si la foto no se puede evaluar, el sistema no la
usa como evidencia de "no urgente": solo avisa que no se pudo leer y decide con el texto y
las guías. Se comprueba con `assert`.

In [ ]:
_imagen_borrosa = "m4/eval_set/imagenes/img_06.jpg"
assert os.path.exists(_imagen_borrosa), "no se encontro img_06 (foto desenfocada) del eval set"

decision_borrosa, bloque_foto_borrosa, _ = app.consultar(
    "Tengo una mancha rara en la espalda, no se si es grave.", _imagen_borrosa,
)
print(decision_borrosa)
print()
print(bloque_foto_borrosa)

# ojo: el texto real trae markdown ('**no** equivale'), por eso se busca 'equivale'
# y 'no evaluable' por separado en vez del substring exacto 'no equivale'.
assert "no evaluable" in bloque_foto_borrosa.lower(), "la foto borrosa deberia marcar 'No evaluable', no dar una clase"
assert "equivale" in bloque_foto_borrosa.lower(), "falta el aviso de que no equivale a 'no urgente'"
print()
print("OK: la foto no evaluable no se conto como evidencia de 'no_urgente'.")


## 4 · Abrir la interfaz con link público

Esta celda levanta la app y deja impreso abajo un link `https://....gradio.live`. Ábrelo
en el navegador o en el celular; dura mientras esta sesión de Kaggle siga activa.

En la app puedes escribir un síntoma, adjuntar una foto (opcional) o probar los casos de
ejemplo. Por privacidad, no escribas nombres ni datos que identifiquen a una persona.

Cuando termines, detén la sesión de Kaggle (o corre `demo.close()` en una celda nueva)
para no seguir gastando tu cuota semanal de GPU.

In [ ]:
demo = app.construir_app()
demo.launch(share=True, prevent_thread_lock=True)
print()
print("App corriendo en segundo plano -- usa el link https://....gradio.live de arriba.")


---
**Qué quedó corriendo:** los 4 modelos cargados una sola vez en fp16, la verificación con
casos reales de la evaluación y la app con link público.

Cada consulta queda registrada en `m5/logs/consultas.jsonl`, sin el texto que se escribió:
solo si hubo foto, qué herramientas se usaron, la sugerencia, la confianza y el tiempo de
respuesta. Para ver un resumen: `!python m5/observabilidad.py --resumen`.

Proyecto Integrador SI4006, Universidad EAFIT · Equipo: Juan Esteban, Isabella, Camilo y
María Alejandra.